# Faz 5 — Kalibrasyon, Eşik Optimizasyonu ve Seçici Tahmin

**Bölüm B:** ECE, Brier, sıcaklık ölçekleme, sınıf başına F1 eşik opt.
**Bölüm D:** Entropy belirsizliği, risk-kapsama eğrisi, seçici tahmin.

**Giriş:** Faz3c'den kaydedilen fold tahminleri (`.npz`/`.csv` formatında).
**Hold-out skoru:** 5 fold modelinin ensemble ortalaması → eşik sweep.

**Sıra:** Faz3c (5 fold eğitim) → **Faz5** (kalibrasyon + değerlendirme)


In [ ]:
import os, sys
from pathlib import Path

IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
IS_LOCAL  = not IS_COLAB and not IS_KAGGLE

print(f'Colab:{IS_COLAB}  Kaggle:{IS_KAGGLE}  Local:{IS_LOCAL}')

In [ ]:
if IS_COLAB or IS_KAGGLE:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'scikit-learn', 'matplotlib',
                    'python-dotenv', 'openpyxl', 'timm>=0.9'], check=True)
    print('Kurulum ✓')

In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BASE = Path('/content/drive/MyDrive/abdomen')
    WORK_DIR   = DRIVE_BASE / 'outputs'
elif IS_KAGGLE:
    WORK_DIR   = Path('/kaggle/working')
elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass
    _proje   = Path(os.environ.get('TR_ABDOMEN_PROJE', r'D:/makale-pdf/Proje'))
    WORK_DIR = Path(os.environ.get('ABDOMEN_OUT_DIR',  str(_proje / 'outputs')))

CKPT_DIR   = Path(os.environ.get('ABDOMEN_CKPT_DIR', str(WORK_DIR / 'checkpoints')))
PRED_DIR   = CKPT_DIR / 'predictions'
CALIB_DIR  = CKPT_DIR / 'calibration'
CALIB_DIR.mkdir(parents=True, exist_ok=True)

from src.config import SUPER_CLASSES
N_FOLDS  = 5
print(f'CKPT_DIR : {CKPT_DIR}')
print(f'Sınıflar : {SUPER_CLASSES}')

In [ ]:
if IS_COLAB or IS_KAGGLE:
    import subprocess
    REPO_DIR = Path('/content/repo') if IS_COLAB else Path('/kaggle/working/repo')
    if not (REPO_DIR / 'src').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/ramazan2020/abdomen1', str(REPO_DIR)],
            check=True
        )
    sys.path.insert(0, str(REPO_DIR))
else:
    if str(Path('.').resolve()) not in sys.path:
        sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
"""
Faz3c'nin her fold için kaydettiği tahmin dosyalarını yükle.
Beklenen format (Faz3c çıktısı):
  checkpoints/predictions/fold{k}_val.npz
  checkpoints/predictions/fold{k}_holdout.npz
Her .npz içerir:
  'logits'   : (N, 6) float32 — sigmoid öncesi
  'probs'    : (N, 6) float32 — sigmoid sonrası
  'labels'   : (N, 6) int
  'case_ids' : (N,) str
"""
import numpy as np

val_preds      = []
holdout_preds  = []

for fold in range(N_FOLDS):
    vp = PRED_DIR / f'fold{fold}_val.npz'
    hp = PRED_DIR / f'fold{fold}_holdout.npz'
    assert vp.exists(), f'Val tahmin dosyası yok: {vp}\nFaz3c tamamlanmış mı?'
    assert hp.exists(), f'Holdout tahmin dosyası yok: {hp}\nFaz3c tamamlanmış mı?'
    val_preds.append(np.load(vp))
    holdout_preds.append(np.load(hp))
    print(f'Fold {fold}: val={len(val_preds[-1]["probs"]):,}  holdout={len(holdout_preds[-1]["probs"]):,}')

# Val: fold validasyon setleri birleştir (kalibrasyon için)
val_logits = np.concatenate([v['logits'] for v in val_preds], axis=0)
val_probs  = np.concatenate([v['probs']  for v in val_preds], axis=0)
val_labels = np.concatenate([v['labels'] for v in val_preds], axis=0)
print(f'\nToplam val  : {len(val_probs):,} vaka')

In [ ]:
"""
Hold-out ensemble: 5 fold modelinin olasılıklarını eşit ağırlıkla ortala.
Bu, yayın standardı değerlendirme yaklaşımıdır (tek modele göre daha sağlam).
"""
from src.calibration import ensemble_average

ho_probs_list  = [hp['probs']  for hp in holdout_preds]
ho_logits_list = [hp['logits'] for hp in holdout_preds]
ho_labels      = holdout_preds[0]['labels']  # tüm fold'larda aynı

ho_probs_ens  = ensemble_average(ho_probs_list)
ho_logits_ens = ensemble_average(ho_logits_list)

print(f'Hold-out vaka sayısı : {len(ho_probs_ens):,}')
print(f'Ensemble probs shape : {ho_probs_ens.shape}')

In [ ]:
"""ECE, Brier ve güvenilirlik diyagramı — kalibrasyon ÖNCE."""
import matplotlib.pyplot as plt
from src.calibration import ece_score, brier_score, reliability_diagram

ece_before   = ece_score(val_probs, val_labels)
brier_before = brier_score(val_probs, val_labels)
print(f'Kalibrasyon ÖNCESİ (val set):')
print(f'  ECE   = {ece_before:.4f}')
print(f'  Brier = {brier_before:.4f}')

# Güvenilirlik diyagramı (tüm sınıflar)
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for c, ax in enumerate(axes.flat):
    bc, ba, bn = reliability_diagram(val_probs, val_labels, class_idx=c)
    ax.bar(bc, ba, width=1/15, alpha=0.7, label='Accuracy')
    ax.bar(bc, bc, width=1/15, alpha=0.3, color='red', label='Perfect calib')
    ax.set_title(SUPER_CLASSES[c], fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    if c == 0: ax.legend(fontsize=7)
plt.suptitle('Güvenilirlik Diyagramı — Kalibrasyon Öncesi', fontsize=11)
plt.tight_layout()
plt.savefig(CALIB_DIR / 'reliability_before.png', dpi=150)
plt.show()
print('Kaydedildi ✓')

In [ ]:
"""Sıcaklık ölçekleme — val set'te fit, holdout'a uygula."""
from src.calibration import TemperatureScaler

scaler = TemperatureScaler(init_T=1.0)
T_opt  = scaler.fit(val_logits, val_labels)
print(f'Optimal sıcaklık T = {T_opt:.3f}')

# Kalibrasyon sonrası ECE / Brier (val)
val_probs_cal = scaler.calibrate(val_logits)
ece_after     = ece_score(val_probs_cal, val_labels)
brier_after   = brier_score(val_probs_cal, val_labels)
print(f'\nKalibrasyon SONRASI (val set):')
print(f'  ECE   = {ece_after:.4f}  (delta={ece_before - ece_after:+.4f})')
print(f'  Brier = {brier_after:.4f}  (delta={brier_before - brier_after:+.4f})')

# Hold-out kalibre olasılıkları
ho_probs_cal = scaler.calibrate(ho_logits_ens)

# Checkpoint kaydet
scaler.save(CALIB_DIR / 'temperature_scaler.pt')
print(f'\nSıcaklık ölçekleyici kaydedildi ✓')

# Güvenilirlik diyagramı sonrası
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for c, ax in enumerate(axes.flat):
    bc, ba, bn = reliability_diagram(val_probs_cal, val_labels, class_idx=c)
    ax.bar(bc, ba, width=1/15, alpha=0.7)
    ax.bar(bc, bc, width=1/15, alpha=0.3, color='red')
    ax.set_title(SUPER_CLASSES[c], fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
plt.suptitle(f'Güvenilirlik Diyagramı — Sonrası (T={T_opt:.2f})', fontsize=11)
plt.tight_layout()
plt.savefig(CALIB_DIR / 'reliability_after.png', dpi=150)
plt.show()

In [ ]:
"""Sınıf başına F1 eşik optimizasyonu — val set'te optimize, holdout'ta raporla."""
import json
from sklearn.metrics import classification_report
from src.calibration import find_optimal_thresholds, apply_thresholds

# Val set'te optimal eşikler
thresholds = find_optimal_thresholds(val_probs_cal, val_labels,
                                      class_names=SUPER_CLASSES)
print('Optimal eşikler (val→F1 maks):')
for name, t in thresholds.items():
    print(f'  {name:35s}: {t:.3f}')

# Hold-out değerlendirmesi (kalibre + eşik uygulandıktan sonra)
ho_preds = apply_thresholds(ho_probs_cal, thresholds, SUPER_CLASSES)

print('\n── Hold-out sonuçları (ensemble + kalibrasyon + optimal eşik) ──')
from sklearn.metrics import f1_score, roc_auc_score
macro_f1  = f1_score(ho_labels, ho_preds, average='macro', zero_division=0)
micro_f1  = f1_score(ho_labels, ho_preds, average='micro', zero_division=0)
try:
    auc = roc_auc_score(ho_labels, ho_probs_cal, average='macro')
except ValueError:
    auc = float('nan')
print(f'Macro F1  : {macro_f1:.4f}')
print(f'Micro F1  : {micro_f1:.4f}')
print(f'Macro AUC : {auc:.4f}')
print()
print(classification_report(ho_labels, ho_preds, target_names=SUPER_CLASSES,
                             zero_division=0))

# Kaydet
with open(CALIB_DIR / 'optimal_thresholds.json', 'w') as f:
    json.dump(thresholds, f, indent=2)
with open(CALIB_DIR / 'holdout_metrics.json', 'w') as f:
    json.dump({'macro_f1': macro_f1, 'micro_f1': micro_f1, 'macro_auc': auc}, f)
print('\nEşikler ve metrikler kaydedildi ✓')

In [ ]:
"""
Seçici tahmin — risk-kapsama eğrisi (Bölüm D).
Yüksek belirsizlikli vakalar dışarıda bırakılır; kalan vakalarda hata düşer.
"""  
import matplotlib.pyplot as plt
from src.calibration import entropy_uncertainty, risk_coverage_curve

# Belirsizlik skorları
uncertainty = entropy_uncertainty(ho_probs_cal)
print(f'Belirsizlik istatistikleri:')
print(f'  Ortalama : {uncertainty.mean():.4f}')
print(f'  %90 perc : {np.percentile(uncertainty, 90):.4f}')
print(f'  Maks     : {uncertainty.max():.4f}')

# Risk-kapsama eğrisi
coverage, risk, n_cov = risk_coverage_curve(
    ho_probs_cal, ho_labels,
    uncertainty=uncertainty,
    class_names=SUPER_CLASSES,
    n_thresholds=100,
)

# Belirli kapsama düzeylerinde risk
for frac in [1.0, 0.9, 0.8, 0.7, 0.5]:
    idx = np.argmin(np.abs(coverage - frac))
    print(f'  Kapsama {frac:.0%}: risk={risk[idx]:.4f}  n={n_cov[idx]}')

# Çiz
plt.figure(figsize=(7, 5))
plt.plot(coverage, risk, 'b-', linewidth=2)
plt.xlabel('Kapsama (coverage)')
plt.ylabel('Risk (1 - macro F1)')
plt.title('Seçici Tahmin: Risk-Kapsama Eğrisi')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CALIB_DIR / 'risk_coverage.png', dpi=150)
plt.show()

# Eğri verisi kaydet (tez tablo/şekil için)
np.savez(CALIB_DIR / 'risk_coverage.npz',
         coverage=coverage, risk=risk, n_covered=n_cov)
print('Risk-kapsama eğrisi kaydedildi ✓')

In [ ]:
"""
Triyaj değerlendirmesi — Bölüm D.
Skor: patient_probs.max() → <0.3 normal, 0.3-0.7 gözlem, >0.7 acil.
Gerçek triage label: Bilgi.xlsx'ten türetilir (en az bir patoloji = acil/gözlem).
"""
import numpy as np
from sklearn.metrics import confusion_matrix

triage_score = ho_probs_cal.max(axis=1)   # (N,)

# Triyaj seviyeleri
triage_pred = np.where(triage_score > 0.7, 2,
              np.where(triage_score > 0.3, 1, 0))  # 0=normal,1=gözlem,2=acil

# Gerçek etiket: herhangi bir pozitif patoloji = en az 1
any_pos     = (ho_labels.sum(axis=1) > 0).astype(int)  # 0=negatif, 1=pozitif
triage_true = np.where(any_pos == 0, 0, 1)             # basit 2-sınıf
triage_binary = (triage_pred >= 1).astype(int)          # 1=gözlem veya acil

# Hassasiyet / özgüllük (patoloji tespit perspektifinden)
from sklearn.metrics import classification_report as cr
print('── Triyaj sonuçları (normal vs gözlem/acil) ──')
print(cr(triage_true, triage_binary,
         target_names=['Normal', 'Patoloji'], zero_division=0))

# Triyaj skoru dağılımı
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.hist(triage_score[any_pos == 0], bins=30, alpha=0.6, label='Negatif')
plt.hist(triage_score[any_pos == 1], bins=30, alpha=0.6, label='Pozitif')
plt.axvline(0.3, color='orange', linestyle='--', label='Gözlem eşiği')
plt.axvline(0.7, color='red',    linestyle='--', label='Acil eşiği')
plt.xlabel('Triyaj skoru'); plt.ylabel('Frekans')
plt.title('Triyaj Skoru Dağılımı')
plt.legend(); plt.tight_layout()
plt.savefig(CALIB_DIR / 'triage_distribution.png', dpi=150)
plt.show()

# Özet istatistikler
import json
n_total  = len(triage_score)
n_normal = (triage_pred == 0).sum()
n_obs    = (triage_pred == 1).sum()
n_urgent = (triage_pred == 2).sum()
print(f'\nTriyaj dağılımı (hold-out):')
print(f'  Normal  (≤0.3) : {n_normal:4d} ({100*n_normal/n_total:.1f}%)')
print(f'  Gözlem (0.3-0.7): {n_obs:4d} ({100*n_obs/n_total:.1f}%)')
print(f'  Acil   (>0.7)  : {n_urgent:4d} ({100*n_urgent/n_total:.1f}%)')

In [ ]:
"""
Etiket-verimliliği ablasyon özeti — Bölüm C (Tablo 5).
Bu hücre, Faz3c'nin 25%/50%/100% koşullarından kaydedilen sonuçları birleştirir.
Faz3c'de her koşul için ayrı checkpoint kaydedilmiş olmalı.
"""
import pandas as pd

label_fracs = [0.25, 0.50, 1.00]
conditions  = [
    'scratch',        # ImageNet yok, CT-MAE yok
    'imagenet',       # ImageNet ön-eğitim, CT-MAE yok
    'ctmae',          # CT-MAE ön-eğitim (ana katkı K4)
]

rows = []
for cond in conditions:
    for frac in label_fracs:
        frac_str = f'{int(frac*100)}pct'
        metrics_file = CKPT_DIR / 'ablation' / f'{cond}_{frac_str}_metrics.json'
        if metrics_file.exists():
            with open(metrics_file) as f:
                m = json.load(f)
            rows.append({'Koşul': cond, 'Etiket %': f'{int(frac*100)}%',
                         'Macro F1': m.get('macro_f1', float('nan')),
                         'AUC':      m.get('macro_auc', float('nan'))})
        else:
            rows.append({'Koşul': cond, 'Etiket %': f'{int(frac*100)}%',
                         'Macro F1': float('nan'), 'AUC': float('nan')})

df = pd.DataFrame(rows)
print('Etiket-verimliliği ablasyon tablosu (Tablo 5):')
print(df.to_string(index=False, float_format='{:.3f}'.format))

df.to_csv(CALIB_DIR / 'label_efficiency_table.csv', index=False)
print('\nTablo kaydedildi ✓')